In [ ]:
from collections import namedtuple
import json
import numpy as np
import pandas as pd
from scipy.stats import beta as beta_dist
import mlflow
import matplotlib.pyplot as plt

In [ ]:
rng = np.random.default_rng(seed=2)

In [ ]:
mlflow.set_experiment("baseline_vs_adaptive")

In [ ]:
n_rounds = 15_000
rolling_round = 150

In [ ]:
ArmParams = namedtuple("ArmParams", ["alpha", "beta"])

# Data

In [ ]:
df = pd.read_csv("../data/raw/bank-additional-full.csv", sep=";")

In [ ]:
df["reward"] = (df["y"] == "yes").astype(int)

In [ ]:
arms = df["contact"].unique()

# Baseline

In [ ]:
arm_conv_rates = df.groupby(by=["contact"])["reward"].mean()
display(arm_conv_rates)

In [ ]:
baseline_arm = max(arms, key=arm_conv_rates.get)
display(baseline_arm)

In [ ]:
with mlflow.start_run(run_name="baseline_replay"):
    mlflow.log_param("policy", "fixed_best_historical_arm")
    mlflow.log_param("arm", baseline_arm)
    mlflow.log_param("evaluation_method", "replay_per_arm_pool")
    mlflow.log_param("n_rounds", n_rounds)

    pool = df.loc[df["contact"] == baseline_arm, "reward"].to_numpy()
    rewards = []
    for _ in range(n_rounds):
        reward = rng.choice(pool)
        rewards.append(reward)

    baseline_rewards = np.array(rewards)
    baseline_rewards_cumulative = np.cumsum(baseline_rewards)

    mlflow.log_metric("conversion_rate", baseline_rewards.mean())
    mlflow.log_metric("total_reward", int(baseline_rewards_cumulative[-1]))

# Adaptive

In [ ]:
prior_alpha = 1.0
prior_beta = 1.0
arm_params = {arm: ArmParams(alpha=prior_alpha, beta=prior_beta) for arm in arms}

In [ ]:
with mlflow.start_run(run_name="thompson_sampling_no_context_replay"):
    mlflow.log_param("policy", "thompson_sampling_no_context")
    mlflow.log_param("prior_alpha", prior_alpha)
    mlflow.log_param("prior_beta", prior_beta)
    mlflow.log_param("evaluation_method", "replay_per_arm_pool")
    mlflow.log_param("n_rounds", n_rounds)

    pools = {arm: df.loc[df["contact"] == arm, "reward"].to_numpy() for arm in arms}
    rewards = []
    for r in range(n_rounds):
        # Exploration x Exploitation
        arm_probs = {arm: rng.beta(arm_params[arm].alpha, arm_params[arm].beta) for arm in arms}
        chosen_arm = max(arms, key=arm_probs.get)

        reward = rng.choice(pools[chosen_arm])
        rewards.append(reward)

        if r % rolling_round == 0:
            mlflow.log_metric("rolling_conversion_rate", np.mean(rewards[-rolling_round:]), step=r)

        # Atualização parâmetros (ir na direção de exploitation)
        if reward == 1:
            arm_params[chosen_arm] = ArmParams(
                alpha=arm_params[chosen_arm].alpha + 1,
                beta=arm_params[chosen_arm].beta
            )
        else:
            arm_params[chosen_arm] = ArmParams(
                alpha=arm_params[chosen_arm].alpha,
                beta=arm_params[chosen_arm].beta + 1
            )

    ts_rewards = np.array(rewards)
    ts_rewards_cumulative = np.cumsum(ts_rewards)

    for arm in arms:
        mlflow.log_metric(f"posterior_alpha_{arm}", arm_params[arm].alpha)
        mlflow.log_metric(f"posterior_beta_{arm}", arm_params[arm].beta)
    mlflow.log_metric("conversion_rate", ts_rewards.mean())
    mlflow.log_metric("total_reward", int(ts_rewards_cumulative[-1]))

# Comparison

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(baseline_rewards_cumulative, label=f"Baseline ({baseline_arm})")
plt.plot(ts_rewards_cumulative, label="Thompson Sampling")
plt.xlabel("Rounds")
plt.ylabel("Conversões acumuladas")
plt.title("Baseline vs Adaptive (Thompson Sampling)")
plt.legend()
plt.show()

In [ ]:
x = np.linspace(0, 1, 1000)

for arm in arms:
    plt.figure(figsize=(8, 5))
    plt.plot(x, beta_dist.pdf(x, prior_alpha, prior_beta), label="Prior", linestyle="--", color="gray")
    plt.plot(x, beta_dist.pdf(x, arm_params[arm].alpha, arm_params[arm].beta), label="Posterior", linestyle="-", color="blue")
    plt.axvline(arm_conv_rates[arm], label="Taxa de conversão (histórica)", linestyle=":", color="red")
    plt.title(f"{arm} — Prior vs Posterior")
    plt.legend()
    plt.show()

# Persistence

In [ ]:
arm_params_export = {arm: {"alpha": params.alpha, "beta": params.beta} for arm, params in arm_params.items()}

with open("../../api/artifacts/non_contextual_bandit.json", "w") as f:
    json.dump(arm_params_export, f, indent=2)